In [0]:
df_bronze = spark.read.table("workspace.default.bronze_nyctaxi")
df_silver = spark.read.table("workspace.default.silver_nyctaxi")
df_gold = spark.read.table("workspace.default.gold_faturamento_diario")

In [0]:
# lista onde vamos guardar o resultado de cada teste: (nome do teste, passou ou não, detalhe)
resultados = []

def checar(nome, condicao, detalhe=""):
    """
    Função auxiliar que registra o resultado de um teste.
    nome: descrição curta do que está sendo testado
    condicao: um booleano (True = passou, False = falhou)
    detalhe: texto extra pra ajudar a debugar se falhar
    """
    resultados.append((nome, condicao, detalhe))
    status = "✅ PASSOU" if condicao else "❌ FALHOU"
    print(f"{status} — {nome} {detalhe}")

In [0]:
from pyspark.sql.functions import col

total_bronze = df_bronze.count()

checar(
    "Bronze não está vazio",
    total_bronze > 0,
    f"(total: {total_bronze} linhas)"
)

nulos_pickup = df_bronze.filter(col("tpep_pickup_datetime").isNull()).count()

checar(
    "Bronze sem nulos em tpep_pickup_datetime",
    nulos_pickup == 0,
    f"(encontrados: {nulos_pickup} nulos)"
)

In [0]:
total_silver = df_silver.count()

checar(
    "Silver não tem mais linhas que Bronze",
    total_silver <= total_bronze,
    f"(silver: {total_silver}, bronze: {total_bronze})"
)

fare_negativo = df_silver.filter(col("fare_amount") <= 0).count()

checar(
    "Silver sem fare_amount zero ou negativo",
    fare_negativo == 0,
    f"(encontrados: {fare_negativo} linhas inválidas)"
)

distancia_invalida = df_silver.filter(col("trip_distance") <= 0).count()

checar(
    "Silver sem trip_distance zero ou negativo",
    distancia_invalida == 0,
    f"(encontrados: {distancia_invalida} linhas inválidas)"
)

In [0]:
from pyspark.sql.functions import sum as _sum

faturamento_negativo = df_gold.filter(col("faturamento_total") < 0).count()

checar(
    "Gold sem faturamento_total negativo",
    faturamento_negativo == 0,
    f"(encontrados: {faturamento_negativo} dias com valor negativo)"
)

dias_sem_corrida = df_gold.filter(col("total_corridas") <= 0).count()

checar(
    "Gold sem dias com total_corridas zero ou negativo",
    dias_sem_corrida == 0,
    f"(encontrados: {dias_sem_corrida} dias inválidos)"
)

datas_duplicadas = df_gold.groupBy("Data").count().filter(col("count") > 1).count()

checar(
    "Gold sem datas duplicadas (uma linha por dia)",
    datas_duplicadas == 0,
    f"(encontradas: {datas_duplicadas} datas repetidas)"
)